# 07 · Main Models

So sánh mô hình chính cho bài toán phân loại 11 lớp. Dùng `train_fe.csv`, các đặc trưng được chọn từ `feature_cols_fe.csv` và GroupKFold theo `group_id`. Chọn mô hình theo macro-F1 trung bình; không sử dụng tập test để lựa chọn. Các ứng viên gồm Random Forest, Extra Trees và HistGradientBoosting, đều có seed cố định.

In [1]:
from pathlib import Path
from datetime import datetime
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score, accuracy_score

SEED = 42
RUN_START = datetime.now()
# Sửa đường dẫn này nếu bộ dữ liệu nằm ngoài thư mục dự án.
# Dữ liệu tham chiếu tại thư mục Challenge 2 trên Desktop.
ROOT = Path(r"C:\Users\Welcome\Desktop\ML cuối kỳ\MAY_HOC_NHOM\Challenge\Challenge 2")
DATA_DIR = ROOT / "DATA_CHALLENGE2"
PROC_DIR = DATA_DIR / "processed"
LOG_PATH = ROOT / "logs" / "experiment_log.csv"
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.base import clone

TRAIN_PATH = PROC_DIR / "train_fe.csv"
COLS_PATH = PROC_DIR / "feature_cols_fe.csv"
if not TRAIN_PATH.exists():
    raise FileNotFoundError(f"Không tìm thấy {TRAIN_PATH}. Đặt CSV theo cấu trúc DATA_CHALLENGE2/processed hoặc cập nhật DATA_DIR.")
train = pd.read_csv(TRAIN_PATH)
TARGET, GROUP = "Class", "group_id"
feature_cols = pd.read_csv(COLS_PATH).iloc[:, 0].dropna().astype(str).tolist() if COLS_PATH.exists() else [c for c in train if c not in ("Id", TARGET, GROUP)]
X, y, groups = train[feature_cols], train[TARGET], train[GROUP]
cv = GroupKFold(n_splits=5)
models = {
    "RandomForest": RandomForestClassifier(n_estimators=400, min_samples_leaf=2, class_weight="balanced_subsample", n_jobs=-1, random_state=SEED),
    "ExtraTrees": ExtraTreesClassifier(n_estimators=400, min_samples_leaf=2, class_weight="balanced", n_jobs=-1, random_state=SEED),
    "HistGradientBoosting": make_pipeline(SimpleImputer(strategy="median"), HistGradientBoostingClassifier(max_iter=250, learning_rate=0.08, l2_regularization=1.0, random_state=SEED)),
}
rows=[]
for name, estimator in models.items():
    for fold, (tr, va) in enumerate(cv.split(X, y, groups), start=1):
        model=clone(estimator)
        model.fit(X.iloc[tr], y.iloc[tr])
        pred=model.predict(X.iloc[va])
        rows.append({"model":name,"fold":fold,"macro_f1":f1_score(y.iloc[va],pred,average="macro"),"accuracy":accuracy_score(y.iloc[va],pred)})
fold_results=pd.DataFrame(rows)
summary=fold_results.groupby("model").agg(macro_f1_mean=("macro_f1","mean"),macro_f1_std=("macro_f1","std"),accuracy_mean=("accuracy","mean")).sort_values("macro_f1_mean",ascending=False)
display(summary.round(4))
best_model_name=summary.index[0]
print("Chọn theo macro-F1 CV:", best_model_name)

,macro_f1_mean,macro_f1_std,accuracy_mean
model,,,
HistGradientBoosting,0.6087,0.0081,0.5750
RandomForest,0.5980,0.0103,0.5649
ExtraTrees,0.5734,0.0084,0.5266


Chọn theo macro-F1 CV: HistGradientBoosting


## Ghi log kết quả

In [2]:
from datetime import datetime
import csv, platform, sys
LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
now=datetime.now()
elapsed_sec=round((now-RUN_START).total_seconds(),1)
run_id=f"07_Main_Models_{now:%Y%m%d_%H%M%S}"
summary_text="; ".join(f"{name}: macro_F1={row.macro_f1_mean:.6f} ± {row.macro_f1_std:.6f}, accuracy={row.accuracy_mean:.6f}" for name,row in summary.iterrows())
record={"run_id":run_id,"date":now.strftime("%Y-%m-%d"),"time":now.strftime("%H:%M:%S"),
 "runner":"Ngô Thanh Phong","stage":"07_Main_Models","notebook":"07_Main_Models.ipynb",
 "goal":"So sánh mô hình chính bằng GroupKFold theo group_id",
 "steps":summary_text,"duration_sec":elapsed_sec,"python":sys.version.split()[0],"platform":platform.platform(),
 "notes":f"Chọn {best_model_name} theo macro-F1 CV; test không dùng để chọn mô hình"}
fields=["run_id","date","time","runner","stage","notebook","goal","steps","duration_sec","python","platform","notes"]
exists=LOG_PATH.exists() and LOG_PATH.stat().st_size>0
with LOG_PATH.open("a",newline="",encoding="utf-8-sig") as f:
 writer=csv.DictWriter(f,fieldnames=fields)
 if not exists: writer.writeheader()
 writer.writerow(record)

md_path=LOG_PATH.parent / f"{record['stage']}_{now:%Y%m%d_%H%M%S}_Tô-Doanh-Quang-Khu.md"
suffix=2
while md_path.exists():
    md_path=LOG_PATH.parent / f"{record['stage']}_{now:%Y%m%d_%H%M%S}_Tô-Doanh-Quang-Khu_{suffix}.md"
    suffix+=1
start_text=RUN_START.strftime("%H:%M:%S")
end_text=now.strftime("%H:%M:%S")
steps_md="\n".join(f"- {part.strip()}" for part in record["steps"].split(";"))
md_text=f"""# Nhật ký thí nghiệm – {record['stage']}

| Mục | Giá trị |
|---|---|
| Người chạy | {record['runner']} |
| Ngày chạy | {now.strftime('%d/%m/%Y')} |
| Giờ bắt đầu – kết thúc | {start_text} – {end_text} ({elapsed_sec} giây) |
| Notebook | `{record['notebook']}` |
| Mục tiêu | {record['goal']} |
| Môi trường | Python {sys.version.split()[0]} · {platform.platform()} |

## Các bước & kết quả

{steps_md}

## Nhận xét

{record['notes']}
"""
md_path.write_text(md_text,encoding="utf-8")

print(f"Đã ghi CSV: {LOG_PATH} | Markdown: {md_path}")

Đã ghi CSV: C:\Users\Welcome\Desktop\ML cuối kỳ\MAY_HOC_NHOM\Challenge\Challenge 2\logs\experiment_log.csv | Markdown: C:\Users\Welcome\Desktop\ML cuối kỳ\MAY_HOC_NHOM\Challenge\Challenge 2\logs\07_Main_Models_20261112_185334_Tô-Doanh-Quang-Khu.md


## Fit mô hình được chọn trên toàn bộ train

Ô tùy chọn: fit mô hình thắng CV lên toàn bộ tập train và tạo dự đoán test nếu `test_fe.csv` có sẵn. Chưa ghi submission để tránh vô tình ghi đè dữ liệu đầu vào.

In [3]:
TEST_PATH=PROC_DIR / "test_fe.csv"
if TEST_PATH.exists():
    test=pd.read_csv(TEST_PATH)
    best_estimator=clone(models[best_model_name])
    best_estimator.fit(X,y)
    predictions=best_estimator.predict(test[feature_cols])
    output=pd.DataFrame({"Id":test["Id"] if "Id" in test else np.arange(len(test)),"Class":predictions})
    out_path=ROOT / "outputs" / "07_main_models_predictions.csv"
    out_path.parent.mkdir(parents=True,exist_ok=True)
    output.to_csv(out_path,index=False)
    print("Đã lưu dự đoán:",out_path,"|",output.shape)
else:
    print("test_fe.csv chưa có; bỏ qua fit/predict test.")

Đã lưu dự đoán: C:\Users\Welcome\Desktop\ML cuối kỳ\MAY_HOC_NHOM\Challenge\Challenge 2\outputs\07_main_models_predictions.csv | (3600, 2)
